# DiffusionGemma-Jev (`djev`) on Google Colab Pro (`A100` / `L4`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taeold/djev-run/blob/main/colab.ipynb)

> **Colab Pro `A100` / `L4` Required by Default:** This notebook requests an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** (`gpuClass: premium`, `machine_shape: hm`) so all `17.53 GiB` of weights stay 100% in GPU VRAM (`~45 ms` on A100 HBM2e, `~65 ms` on L4). If Colab connects you to a default `T4`, click **`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU or L4 GPU`** (or top-right dropdown arrow next to `T4 -> Change runtime type`).

Run **DiffusionGemma-Jev** (`nvidia/diffusiongemma-26B-A4B-it-NVFP4`, `26B` total parameters, `4B` active per token across `128` experts) directly inside Google Colab Pro on an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** GPU using standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions` with `extra_body.vllm_xargs`), and play the built-in 1-step diffusion games (`/tetris`, `/dino`, and `/snake`) live inside the notebook.

| Colab Runtime | GPU / Compute Capability | Usable VRAM | `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) | Active vLLM Kernel Path |
| :--- | :--- | :--- | :--- | :--- |
| **Colab Pro (`A100`)** | NVIDIA A100 (`SM 8.0`) | `40.0 GiB` / `80.0 GiB` | **Recommended (`~45 ms/step`, 100% VRAM)** (`KV_CACHE_GB=8`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Pro (`L4`)** | NVIDIA L4 (`SM 8.9`) | `22.5 GiB` (`24 GB`) | **Supported (`~65 ms/step`, 100% VRAM)** (`KV_CACHE_GB=1.5`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Free (`T4`)** | NVIDIA T4 (`SM 7.5`) | `15.0 GiB` (`16 GB`) | **Blocked by default** (`ALLOW_SLOW_T4_OFFLOAD = False`; requires `5 GB` PCIe CPU offload at `~400-600 ms/step`) | Switch to `A100` or `L4` in `Runtime -> Change runtime type` |

In [1]:
import os
import subprocess

ALLOW_SLOW_T4_OFFLOAD = False

try:
    smi_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()[0]
    gpu_name, total_mib, free_mib, compute_cap = [x.strip() for x in smi_out.split(",")]
    vram_gib = float(total_mib) / 1024.0
    free_gib = float(free_mib) / 1024.0
    sm_major, sm_minor = [int(x) for x in compute_cap.split(".")]
except Exception as e:
    raise RuntimeError(
        "No GPU detected via nvidia-smi. In Colab, click Runtime -> Change runtime type -> "
        "Hardware accelerator -> select A100 GPU or L4 GPU."
    ) from e

if vram_gib < 22.0 and not ALLOW_SLOW_T4_OFFLOAD:
    raise RuntimeError(
        "Detected NVIDIA T4 (15 GB). T4 requires 5 GB PCIe CPU offload (~400-600 ms/step). "
        "Please switch to an A100 or L4 GPU via: Runtime -> Change runtime type "
        "-> Hardware accelerator -> A100 GPU or L4 GPU (or set ALLOW_SLOW_T4_OFFLOAD = True in this cell)."
    )

if vram_gib < 22.0:
    KV_CACHE_GB = 0.5
    CPU_OFFLOAD_GB = 5.0
    GPU_UTIL = 0.85
    DTYPE = "float16"
elif vram_gib < 30.0:
    KV_CACHE_GB = 1.5
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.90
    DTYPE = "auto"
elif vram_gib < 60.0:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.85
    DTYPE = "auto"
else:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.40
    DTYPE = "auto"

os.environ["KV_CACHE_GB"] = str(KV_CACHE_GB)
os.environ["CPU_OFFLOAD_GB"] = str(CPU_OFFLOAD_GB)
os.environ["GPU_UTIL"] = str(GPU_UTIL)
os.environ["DTYPE"] = DTYPE
os.environ["CANVAS"] = "256"
os.environ["DISABLE_MM"] = "1"

print(f"GPU Device         : {gpu_name} (SM {sm_major}.{sm_minor})")
print(f"Total / Free VRAM  : {vram_gib:.2f} GiB total / {free_gib:.2f} GiB free (0.00 GiB used by notebook kernel)")
print(f"Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)")
print(f"MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype={DTYPE}, gpu_util={GPU_UTIL}, cpu_offload_gb={CPU_OFFLOAD_GB})")
print(f"Configured KV Cache: {KV_CACHE_GB} GiB (CANVAS=256, DISABLE_MM=1)")

GPU Device         : NVIDIA A100-SXM4-40GB (SM 8.0)
Total / Free VRAM  : 39.56 GiB total / 39.56 GiB free (0.00 GiB used by notebook kernel)
Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)
MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype=auto, gpu_util=0.85, cpu_offload_gb=0.0)
Configured KV Cache: 8.0 GiB (CANVAS=256, DISABLE_MM=1)


## Step 1: Install `djev-run` (`uv pip install`) & Start Standalone `vLLM` Server (`entrypoint.sh`)

The cell below creates an isolated Python 3.12 virtualenv (`/opt/djev_venv`) via `uv`, installs the official `vLLM` manylinux wheel (`wheels.vllm.ai`) and NVIDIA's `cuda-compat-13-0` forward-compatibility driver package, downloads `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) into `/dev/shm/dgemma` (or `/content/dgemma` with a `/dev/shm/dgemma` symlink if `/dev/shm` has `< 22 GiB` free), and launches the standalone `vLLM` server (`entrypoint.sh`) on `http://127.0.0.1:8080`.

In [2]:
import glob
import json
import os
import shutil
import subprocess
import time
import urllib.request

DJEV_PORT = 8080
DJEV_BASE_URL = f"http://127.0.0.1:{DJEV_PORT}"
VLLM_WHEEL = "https://wheels.vllm.ai/dee37d89115db4c94a820a79a78a7828e141c910/vllm-0.29.1rc1.dev347%2Bgdee37d891-cp38-abi3-manylinux_2_28_x86_64.whl"
CUDA_COMPAT_DEB = "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-compat-13-0_580.95.05-0ubuntu1_amd64.deb"

def install_and_start_djev():
    for p in (DJEV_PORT, 8898):
        try:
            r = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{p}/health", timeout=2).read().decode())
            if r.get("status") == "ok":
                print(f"[djev] Connected to active in-process vLLM server at http://127.0.0.1:{p}")
                return f"http://127.0.0.1:{p}"
        except Exception:
            pass

    uv_bin = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
    if not os.path.exists(uv_bin):
        print("[djev] Installing uv package manager...", flush=True)
        subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)

    if not os.path.exists("/entrypoint.sh"):
        print("[djev] Creating Python 3.12 virtualenv at /opt/djev_venv and installing vLLM wheel...", flush=True)
        subprocess.run([uv_bin, "venv", "/opt/djev_venv", "--python", "3.12"], check=True)
        subprocess.run([
            uv_bin, "pip", "install",
            "--python", "/opt/djev_venv/bin/python",
            VLLM_WHEEL,
            "accelerate",
            "huggingface_hub[hf_transfer]",
            "--extra-index-url", "https://download.pytorch.org/whl/cu130",
            "--index-strategy", "unsafe-best-match",
        ], check=True)

        if not os.path.exists("/usr/local/cuda-13.0/compat/libcuda.so.1"):
            print("[djev] Installing NVIDIA cuda-compat-13-0 driver compatibility package...", flush=True)
            urllib.request.urlretrieve(CUDA_COMPAT_DEB, "/tmp/cuda-compat-13-0.deb")
            subprocess.run(["dpkg", "-i", "/tmp/cuda-compat-13-0.deb"], check=True)
            os.remove("/tmp/cuda-compat-13-0.deb")

        os.makedirs("/opt/dgemma", exist_ok=True)
        for fname in ("entrypoint.sh", "tetris.html", "dino.html", "snake.html"):
            url = f"https://raw.githubusercontent.com/taeold/djev-run/main/{fname}"
            dest = "/entrypoint.sh" if fname == "entrypoint.sh" else f"/opt/dgemma/{fname}"
            urllib.request.urlretrieve(url, dest)
        os.chmod("/entrypoint.sh", 0o755)

    if not os.path.exists("/dev/shm/dgemma/.ready"):
        shm_free_gib = shutil.disk_usage("/dev/shm").free / (1024 ** 3)
        target_dir = "/dev/shm/dgemma" if shm_free_gib >= 22.0 else "/content/dgemma"
        os.makedirs(target_dir, exist_ok=True)
        if target_dir != "/dev/shm/dgemma":
            if os.path.islink("/dev/shm/dgemma") or os.path.exists("/dev/shm/dgemma"):
                shutil.rmtree("/dev/shm/dgemma", ignore_errors=True)
            os.symlink(target_dir, "/dev/shm/dgemma")
        print(f"[djev] Downloading nvidia/diffusiongemma-26B-A4B-it-NVFP4 (17.53 GiB) into {target_dir}...", flush=True)
        dl_env = os.environ.copy()
        dl_env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
        subprocess.run([
            "/opt/djev_venv/bin/python", "-c",
            f"from huggingface_hub import snapshot_download; snapshot_download(repo_id='nvidia/diffusiongemma-26B-A4B-it-NVFP4', local_dir={target_dir!r})"
        ], env=dl_env, check=True)
        open("/dev/shm/dgemma/.ready", "w").close()

    site_pkg = subprocess.check_output(
        ["/opt/djev_venv/bin/python", "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
        text=True,
    ).strip()
    nv_libs = [p for p in glob.glob(f"{site_pkg}/nvidia/*/lib") if os.path.isdir(p)]
    ld_paths = ["/usr/local/cuda-13.0/compat", f"{site_pkg}/torch/lib"] + nv_libs
    if os.environ.get("LD_LIBRARY_PATH"):
        ld_paths.append(os.environ["LD_LIBRARY_PATH"])
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = ":".join(ld_paths)
    env["PYTHONPATH"] = f"{site_pkg}:/opt/dgemma:" + env.get("PYTHONPATH", "")

    print("[djev] Launching standalone in-process vLLM server on port 8080...", flush=True)
    log_path = "/tmp/djev_server.log"
    log_file = open(log_path, "w")
    proc = subprocess.Popen(["/entrypoint.sh"], stdout=log_file, stderr=subprocess.STDOUT, env=env)
    log_reader = open(log_path, "r")
    t_start = time.time()
    while time.time() - t_start < 300:
        new_lines = log_reader.read()
        if new_lines:
            print(new_lines, end="", flush=True)
        if proc.poll() is not None:
            remaining = log_reader.read()
            if remaining:
                print(remaining, end="", flush=True)
            raise RuntimeError(f"Server process exited early with code {proc.returncode}.")
        try:
            r = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{DJEV_PORT}/health", timeout=2).read().decode())
            if r.get("status") == "ok":
                print(f"[djev] Server ready in {time.time() - t_start:.1f}s at http://127.0.0.1:{DJEV_PORT}", flush=True)
                return f"http://127.0.0.1:{DJEV_PORT}"
        except Exception:
            time.sleep(1.5)
    raise RuntimeError("Server timed out after 300s.")

DJEV_BASE_URL = install_and_start_djev()

[djev] Connected to active in-process vLLM server at http://127.0.0.1:8080


## Step 2: 1-Step Diffusion Canvas Read via Standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions`)

Tokenize an output template (`POST /tokenize`), pin every scaffold and label token (`diffusion_pinned`), leave the answer slots unpinned (`department`, `urgency`, `refund_requested`), and read all three slot probability distributions simultaneously in **1 forward pass** (`diffusion_max_steps: 1`, `diffusion_read_only: True`).

In [3]:
import json
import math
import time
import urllib.request

SCAFFOLD = [100, 45518, 107, 101]  # <thought>\n</thought>
template_str = "department: a\nurgency: 1\nrefund_requested: yes"

# 1. Tokenize template via POST /tokenize
tok_req = urllib.request.Request(
    f"{DJEV_BASE_URL}/tokenize",
    data=json.dumps({"prompt": template_str, "add_special_tokens": False}).encode(),
    headers={"Content-Type": "application/json"},
)
base_ids = json.loads(urllib.request.urlopen(tok_req, timeout=10).read().decode())["tokens"]
full_template = SCAFFOLD + base_ids

# Find unpinned slot positions (each token immediately before a newline 107 or end of template)
unpinned = [i - 1 for i, t in enumerate(full_template) if i >= 4 and t == 107] + [len(full_template) - 1]
pinned = [i for i in range(len(full_template)) if i not in unpinned]
seed_canvas = [full_template[i] if i in pinned else (256000 + i * 131) for i in range(len(full_template))]

# 2. Run 1-step read-only diffusion forward pass via POST /v1/chat/completions
chat_payload = {
    "model": "djev-dgemma",
    "messages": [
        {
            "role": "system",
            "content": (
                "Answer each question about the ticket state with its single label.\n"
                "department: a = billing, b = technical, c = sales\n"
                "urgency: 1 = low, 2 = minor, 3 = locked production access, 4 = complete outage\n"
                "refund_requested: yes or no"
            ),
        },
        {
            "role": "user",
            "content": json.dumps({
                "ticket_id": "TCK-9042",
                "customer_tier": "enterprise",
                "text": "I was double-charged $149.00 on invoice INV-2026-8841 and my production API key is locked. Please refund the duplicate charge.",
            }),
        },
    ],
    "max_tokens": len(full_template) + 1,
    "logprobs": True,
    "top_logprobs": 16,
    "extra_body": {
        "vllm_xargs": {
            "diffusion_seed_canvas": seed_canvas,
            "diffusion_pinned": pinned,
            "diffusion_max_steps": 1,
            "diffusion_read_only": True,
        }
    },
}

t0 = time.time()
req = urllib.request.Request(
    f"{DJEV_BASE_URL}/v1/chat/completions",
    data=json.dumps(chat_payload).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=30).read().decode())
rtt_ms = round((time.time() - t0) * 1000, 1)
srv_ms = round(resp.get("timing_ms", rtt_ms), 1)
content_lp = resp["choices"][0]["logprobs"]["content"]

def slot_softmax(pos, label_map):
    top = {entry["token"].strip(): entry["logprob"] for entry in content_lp[pos]["top_logprobs"]}
    lps = [top.get(lbl, -20.0) for lbl in label_map]
    mx = max(lps)
    ex = [math.exp(x - mx) for x in lps]
    s = sum(ex)
    return {name: ex[i] / s for i, name in enumerate(label_map.values())}

dept_probs = slot_softmax(unpinned[0], {"a": "billing", "b": "technical", "c": "sales"})
urg_probs = slot_softmax(unpinned[1], {"1": "1", "2": "2", "3": "3", "4": "4"})
ref_probs = slot_softmax(unpinned[2], {"yes": "yes", "no": "no"})

best_dept = max(dept_probs, key=dept_probs.get)
exp_urg = sum(int(k) * v for k, v in urg_probs.items())
best_urg = max(urg_probs, key=urg_probs.get)

print(f"1-Step Diffusion Canvas Read Complete in {srv_ms} ms GPU ({rtt_ms} ms total)")
print("-" * 78)
print(f"{'SLOT ID':<18} | {'TYPE':<8} | {'PREDICTION':<22} | {'CONFIDENCE / SCORE'}")
print("-" * 78)
print(f"{'department':<18} | {'choice':<8} | {best_dept:<22} | {dept_probs[best_dept]*100:5.1f}% (probs: {json.dumps({k: round(v, 3) for k, v in dept_probs.items()})})")
print(f"{'urgency':<18} | {'score':<8} | {'level ' + best_urg + '/4':<22} | {exp_urg:.2f} / 4.00 (conf: {urg_probs[best_urg]*100:5.1f}%)")
print(f"{'refund_requested':<18} | {'noul':<8} | {str(ref_probs['yes'] >= 0.5):<22} | P(yes) = {ref_probs['yes']*100:5.1f}%")
print("-" * 78)

1-Step Diffusion Canvas Read Complete in 58.4 ms GPU (112.1 ms total)
------------------------------------------------------------------------------
SLOT ID            | TYPE     | PREDICTION             | CONFIDENCE / SCORE
------------------------------------------------------------------------------
department         | choice   | billing                |  99.7% (probs: {"billing": 0.997, "technical": 0.003, "sales": 0.0})
urgency            | score    | level 3/4              | 3.00 / 4.00 (conf:  99.4%)
refund_requested   | noul     | True                   | P(yes) = 100.0%
------------------------------------------------------------------------------


## Step 3: Play `/tetris`, `/dino`, and `/snake` Live Inside Colab

Set `DEMO = "/tetris"`, `"/dino"`, or `"/snake"` and run the cell below to embed the live game UI served from port `8080` on your Colab GPU.

In [4]:
# Choose a built-in game: "/tetris", "/dino", or "/snake"
DEMO = "/tetris"

try:
    from google.colab import output
    print(f"Embedding {DEMO} from Colab GPU server (port 8080)...")
    output.serve_kernel_port_as_iframe(8080, path=DEMO, height=660)
except ImportError:
    print("Available demo routes on local server:")
    for r in ("/tetris", "/dino", "/snake"):
        print(f"  -> {DJEV_BASE_URL}{r}")

Embedding /tetris from Colab GPU server (port 8080)...
